# LCEL 인터페이스

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `invoke`, `batch`, `stream` 동기 메서드의 차이를 설명하고 적절한 상황에 사용할 수 있어요
2. `ainvoke`, `abatch`, `astream` 비동기 메서드를 Jupyter 환경에서 실행할 수 있어요
3. `asyncio.gather`로 여러 LLM 호출을 병렬 실행해 성능을 높일 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 체인 구성 방법
- Python `async`/`await` 기초 (선택, 섹션 2에서 사용해요)

## 이전 노트북 연결

`02_Chain.ipynb`에서 체인을 구성하는 방법을 배웠어요. 이번에는 그 체인을 다양한 방식으로 실행하는 LCEL(LangChain Expression Language) 표준 인터페이스를 살펴볼게요.

아래 다이어그램은 LCEL 파이프라인에서 파이프 연산자가 데이터를 어떻게 전달하는지 보여줘요.

```mermaid
flowchart LR
    A["입력<br/>{question}"]:::input --> B["PromptTemplate<br/>|"]:::process
    B --> C["ChatOpenAI<br/>|"]:::process
    C --> D["StrOutputParser<br/>최종 출력"]:::output

    A2["invoke<br/>단일 실행"]:::process --> R1["결과 1개"]:::output
    A3["batch<br/>리스트 입력"]:::process --> R2["결과 리스트"]:::output
    A4["stream<br/>청크 스트리밍"]:::process --> R3["토큰 단위<br/>실시간 출력"]:::output
    A5["ainvoke / abatch<br/>/ astream"]:::process --> R4["비동기 결과"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

`Runnable(러너블)` 프로토콜은 LangChain의 대부분 컴포넌트에 구현되어 있어요. 표준 인터페이스를 통해 사용자 정의 체인을 정의하고 일관된 방식으로 호출할 수 있어요.

**동기 메서드**:
- `invoke`: 단일 입력을 처리하고 결과를 반환해요
- `batch`: 여러 입력을 일괄 처리해요
- `stream`: 응답 청크를 실시간으로 스트리밍해요

**비동기 메서드**:
- `ainvoke`, `abatch`, `astream`: 위 메서드의 비동기 버전이에요
- `astream_log`: 최종 응답뿐만 아니라 중간 단계도 스트리밍해요

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import asyncio
import time

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 기본 체인 구성: 프롬프트 → LLM → 문자열 추출
# 아래 섹션에서 invoke, batch, stream 등 다양한 방식으로 이 체인을 호출해요
prompt_template = PromptTemplate.from_template("{question}")
chain = prompt_template | model | StrOutputParser()

## 1. 동기 메서드 (Synchronous Methods)

동기 메서드는 각 작업이 완료될 때까지 다음 작업을 기다려요. 코드 흐름이 단순하고 예측하기 쉬워요.

세 가지 동기 메서드를 순서대로 살펴볼게요: `invoke` → `batch` → `stream`

### 1.1 invoke - 단일 입력 처리

`invoke`는 가장 기본적인 메서드예요. 단일 입력에 대해 체인을 실행하고 결과를 반환해요. 응답이 완전히 완료된 뒤에야 반환돼요.

In [2]:
# ---------------------------------------------------
# invoke(): 단일 입력 처리하고 완성된 응답 반환하기
# ---------------------------------------------------
# invoke()는 응답 전체가 완성될 때까지 블로킹 후 결과를 반환해요

res = chain.invoke({"question": "파이썬에서 함수를 정의하는 방법 알려줘"})
print(res)

파이썬에서 함수를 정의하는 방법은 매우 간단합니다. `def` 키워드를 사용하여 함수를 정의하고, 함수 이름과 괄호 안에 매개변수를 지정한 다음, 콜론(:)으로 끝냅니다. 그 다음에 함수의 본체를 들여쓰기하여 작성합니다. 아래에 기본적인 함수 정의의 예시를 보여드릴게요.

```python
def 함수이름(매개변수1, 매개변수2):
    # 함수의 본체
    결과 = 매개변수1 + 매개변수2
    return 결과  # 결과를 반환
```

### 예시

더하기 기능을 가진 함수를 정의해보겠습니다.

```python
def add(a, b):
    return a + b

# 함수 호출
result = add(3, 5)
print(result)  # 출력: 8
```

### 설명
- `def add(a, b):`는 `add`라는 이름의 함수를 정의하고 `a`와 `b`라는 두 개의 매개변수를 받습니다.
- `return a + b`는 함수의 결과로 두 매개변수의 합을 반환합니다.
- `add(3, 5)`를 호출하면 3과 5의 합인 8이 반환됩니다.

### 추가 사항
- 매개변수는 기본값을 가질 수 있습니다.
- 가변 인자 리스트(`*args`)나 키워드 인자 목록(`**kwargs`)을 사용할 수도 있습니다.

다양한 방식으로 함수를 정의하고 활용해보세요!


### 1.2 batch - 여러 입력 일괄 처리

`batch`는 입력 리스트를 받아 내부적으로 병렬 처리해 결과 리스트를 반환해요. 개별 `invoke()`를 여러 번 호출하는 것보다 빠를 수 있어요.

In [4]:
# ---------------------------------------------------
# batch(): 여러 입력을 리스트로 전달해 한 번에 처리하기
# ---------------------------------------------------
# ============================================================
# TODO: questions 리스트에 원하는 질문을 추가하고 batch()로 실행해봐요
# 힌트: chain.batch([{"question": "..."}, ...]) 형태로 호출해요
# 예상 결과: 각 질문에 대한 응답이 리스트로 반환돼요
# ============================================================

questions = [
  {"question": "파이썬이란 무엇인가?"},
  {"question": "머신러닝이란 무엇인가?"},
  {"question": "딥러닝이란 무엇인가?"}
]
res = chain.batch(questions)
print(f' ==> [Line 15]: \033[38;2;101;57;170m[res]\033[0m({type(res).__name__}) = \033[38;2;197;80;252m{res}\033[0m')


 ==> [Line 15]: [res](list) = ['파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드의 가독성이 뛰어나고, 문법이 간결하여 배우기 쉽고 사용하기 편리한 특징이 있습니다. 또한 다양한 프로그래밍 패러다임(객체 지향, 절차적, 함수형 프로그래밍 등)을 지원합니다.\n\n주요 특징:\n1. **간결한 문법**: 파이썬은 다른 프로그래밍 언어에 비해 코드가 더 간결하고 명확하여 읽기 쉽습니다.\n2. **크로스 플랫폼**: Windows, macOS, Linux 등 다양한 운영 체제에서 실행할 수 있습니다.\n3. **광범위한 라이브러리**: 데이터 분석, 웹 개발, 인공지능, 머신러닝 등 다양한 분야에서 활용할 수 있는 강력한 라이브러리와 프레임워크가 많습니다. 예를 들어, NumPy, pandas, Django, Flask, TensorFlow 등이 있습니다.\n4. **대화형 프로그래밍**: 파이썬은 대화형 셸을 제공하여 즉시 결과를 확인하면서 프로그래밍할 수 있는 환경을 제공합니다.\n5. **활발한 커뮤니티**: 많은 사용자가 있으며, 다양한 자료와 지원을 받을 수 있는 커뮤니티가 존재합니다.\n\n파이썬은 데이터 과학, 웹 개발, 자동화 스크립트 작성, 게임 개발, 네트워크 프로그래밍 등 여러 분야에서 널리 사용되고 있습니다.', '머신러닝(Machine Learning)은 인공지능(AI)의 한 분야로, 컴퓨터가 데이터를 통해 학습하고 경험을 쌓아 자동으로 개선되는 알고리즘을 개발하는 기술입니다. 즉, 명시적으로 프로그래밍하지 않고도 데이터를 분석하고 예측하거나 결정을 내릴 수 있는 능력을 갖추게 하는 것입니다.\n\n머신러닝은 주로 다음과 같은 세 가지 유형으로 나눌 수 있습니다:\n\n1. **지도 학습(Supervised Learning)**: 입력 데이터와 해당하는 정답(레이블)이 주어져 있을 때, 해당 데이터를 기반으로 모델을 학

### 1.3 stream - 스트리밍 응답

`stream`은 LLM이 토큰을 생성할 때마다 즉시 청크(chunk)를 전달해요. 긴 응답을 처리할 때 사용자가 첫 결과를 더 빨리 볼 수 있어요.

> **실무 팁**: 챗봇 UI처럼 응답을 실시간으로 표시해야 하는 경우 `stream`을 사용해요. 배치 처리나 응답 전체를 저장해야 할 때는 `invoke`가 더 적합해요.

In [8]:
# ---------------------------------------------------
# stream(): 토큰이 생성되는 즉시 화면에 출력하기
# ---------------------------------------------------
# ChatGPT 웹 화면에서 글자가 한 글자씩 나타나는 것과 같은 원리예요
# 응답 완성을 기다리지 않으므로 사용자 체감 속도가 향상돼요
stream_result = chain.stream({"question": "파이썬의 특징 세 가지를 정리하세요"})
print(f' ==> [Line 6]: \033[38;2;115;138;137m[stream_result]\033[0m({type(stream_result).__name__}) = \033[38;2;247;141;180m{stream_result}\033[0m')

for chunk in stream_result:
  print(chunk, end="", flush=True)

 ==> [Line 6]: [stream_result](generator) = <generator object RunnableSequence.stream at 0x12dec5120>
파이썬의 특징은 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드의 가독성이 높고 문법이 간단하여 배우기 쉽습니다. 이를 통해 개발자는 더 적은 코드로 더 많은 작업을 수행할 수 있으며, 다른 사람이 작성한 코드를 이해하기도 용이합니다.

2. **다양한 라이브러리와 프레임워크**: 파이썬은 데이터 과학, 웹 개발, 인공지능, 머신러닝 등 다양한 분야에서 사용되는 풍부한 라이브러리와 프레임워크를 제공합니다. 예를 들어, NumPy, Pandas, Flask, Django, TensorFlow 등은 파이썬의 대표적인 패키지입니다.

3. **인터프리터 언어**: 파이썬은 인터프리터 언어로, 코드를 작성한 후 즉시 실행하고 결과를 확인할 수 있습니다. 이는 개발 과정에서 반복적인 테스트와 디버깅을 용이하게 하며, 생산성을 높이는 데 기여합니다.

### 1.4 stream - 청크 수집해 전체 메시지 구성

스트리밍 청크를 변수에 누적하면 전체 응답도 함께 얻을 수 있어요. 실시간 출력과 전체 저장을 동시에 처리하는 패턴이에요.

In [ ]:
# ---------------------------------------------------
# 스트리밍 청크를 누적해 전체 응답 함께 구성하기
# ---------------------------------------------------
# 실시간 출력과 전체 텍스트 저장을 동시에 처리하는 패턴이에요

stream_result = chain.stream({"question": "파이썬의 특징 세 가지를 정리하세요"})
print(f' ==> [Line 6]: \033[38;2;115;138;137m[stream_result]\033[0m({type(stream_result).__name__}) = \033[38;2;247;141;180m{stream_result}\033[0m')

for chunk in stream_result:
  print(chunk, end="", flush=True)


 ==> [Line 6]: [stream_result](generator) = <generator object RunnableSequence.stream at 0x1326dba60>
파이썬의 특징 세 가지는 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드를 작성할 때 사용자가 이해하기 쉽도록 설계된 문법을 가지고 있습니다. 이는 코드의 가독성을 높이고, 개발자가 쉽게 배우고 사용할 수 있게 합니다.

2. **다양한 라이브러리와 프레임워크**: 파이썬은 데이터 분석, 웹 개발, 머신러닝 등 다양한 분야에서 사용할 수 있는 풍부한 라이브러리와 프레임워크를 제공합니다. 예를 들어, NumPy와 Pandas는 데이터 처리에, Django와 Flask는 웹 개발에 유용합니다.

3. **확장성과 호환성**: 파이썬은 C, C++, 자바 등 다른 프로그래밍 언어와 쉽게 통합할 수 있습니다. 이로 인해 기존 시스템과의 호환성을 유지하면서 확장할 수 있어, 다양한 프로젝트에 활용될 수 있습니다. 

이와 같은 특징 덕분에 파이썬은 다양한 분야에서 널리 사용되고 있습니다.

## 2. 비동기 메서드 (Asynchronous Methods)

앞서 동기 메서드를 살펴봤어요. 이제 비동기(async) 메서드를 알아볼게요.

LLM API 호출 시간의 대부분은 **네트워크 응답 대기**예요. 동기 방식에서는 대기하는 동안 아무것도 못 하지만, 비동기 방식에서는 그 시간에 다른 API 호출을 시작할 수 있어요.

| 방식 | 동작 |
|------|------|
| 동기 `invoke` 3회 | A 완료 대기 → B 완료 대기 → C 완료 대기 (총 시간 = A + B + C) |
| 비동기 `ainvoke` 3회 (`asyncio.gather`) | A, B, C 동시 시작 → 가장 느린 것만큼 대기 (총 시간 ≈ max(A, B, C)) |

> **실무 팁**: FastAPI 같은 웹 서버 환경에서는 비동기가 필수예요. 한 사용자의 LLM 응답을 기다리는 동안 다른 사용자 요청도 처리할 수 있기 때문이에요.

### 2.1 ainvoke - 비동기 단일 입력 처리

`ainvoke`는 `invoke`의 비동기 버전이에요. Jupyter 노트북은 이미 실행 중인 이벤트 루프가 있어서 `asyncio.run()` 대신 `await`를 셀에서 직접 사용할 수 있어요.

> **주의**: 일반 Python 스크립트에서는 `asyncio.run(chain.ainvoke(...))` 형태로 실행해야 해요. Jupyter에서만 `await`를 최상위 레벨에서 사용할 수 있어요.

In [3]:
# ---------------------------------------------------
# ainvoke(): 비동기 단일 호출 (Jupyter에서 await 직접 사용)
# ---------------------------------------------------
# Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 await를 셀 최상위에서 사용 가능
# 일반 Python 스크립트에서는 asyncio.run(chain.ainvoke(...)) 형태로 실행해야 해요

res = await chain.ainvoke({"question": "비동기 프로그래밍이란 무엇인가요?"})
print(f' ==> [Line 7]: \033[38;2;86;147;1m[res]\033[0m({type(res).__name__}) = \033[38;2;11;92;110m{res}\033[0m')


 ==> [Line 7]: [res](str) = 비동기 프로그래밍(Asynchronous Programming)이란, 프로그램의 실행 중에 다른 작업들이 동시에 진행될 수 있도록 하는 프로그래밍 기법입니다. 전통적인 동기 프로그래밍에서는 하나의 작업이 완료될 때까지 다음 작업이 대기해야 하지만, 비동기 프로그래밍에서는 작업을 시작한 후 다른 작업을 진행할 수 있기 때문에 전체적인 효율성을 향상시킬 수 있습니다.

### 비동기 프로그래밍의 주요 개념:

1. **콜백(Callback)**: 비동기 작업이 완료되었을 때 호출되는 함수입니다. 예를 들어, API 요청이 완료되면 콜백 함수가 실행되어 결과를 처리할 수 있습니다.

2. **프라미스(Promise)**: 비동기 작업의 결과를 나타내는 객체입니다. 프라미스는 작업이 성공적으로 완료되었는지, 실패했는지를 나타내며, `.then()`, `.catch()` 메소드를 통해 결과나 오류를 처리할 수 있습니다.

3. **async/await**: ES2017에서 도입된 기능으로, 비동기 코드를 작성할 때 더 간편하고 읽기 쉽게 만들어 줍니다. `async` 함수 안에서는 `await` 키워드를 사용하여 프라미스를 기다릴 수 있습니다.

4. **이벤트 루프(Event Loop)**: 자바스크립트와 같은 일부 프로그래밍 언어에서 비동기 작업을 처리하는 메커니즘입니다. 이벤트 루프는 콜 스택과 작업 큐를 관리하여 비동기 작업이 완료되면 해당 콜백을 실행합니다.

### 비동기 프로그래밍의 장점:

- **효율성**: 여러 작업을 동시에 처리할 수 있어 시간 낭비를 줄일 수 있습니다.
- **사용자 경험 개선**: UI가 멈추지 않고, 사용자와의 상호작용이 지속될 수 있습니다.
- **자원 관리**: 서버와의 통신이나 파일 입출력 등에서 비동기 처리를 통해 자원을 효율적으로 활용할 수 있습니다.

비동기 프로그래밍은 웹 개발, 네트워크 프로그래밍 및 기타 여러 분야에서 널리 사용되며, 이를 통해 비동기 함수 호출,

### 2.2 abatch - 비동기 일괄 처리

`abatch`는 `batch`의 비동기 버전이에요. 내부적으로 `asyncio.gather`를 사용해 여러 입력을 동시에 처리해요.

In [10]:
# ---------------------------------------------------
# abatch(): 비동기 일괄 처리 (내부적으로 asyncio.gather 사용)
# ---------------------------------------------------

questions = [
  {"question": "파이썬이란 무엇인가?"},
  {"question": "머신러닝이란 무엇인가?"},
  {"question": "딥러닝이란 무엇인가?"}
]

res = await chain.abatch(questions)
print(f' ==> [Line 11]: \033[38;2;115;25;107m[res]\033[0m({type(res).__name__}) = \033[38;2;254;35;180m{res}\033[0m')



 ==> [Line 11]: [res](list) = ['파이썬(Python)은 고급 프로그래밍 언어로, 1991년 네이선 판 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 읽기 쉬운 문법과 강력한 기능 덕분에 다양한 용도로 널리 사용됩니다. 다음은 파이썬의 주요 특징입니다:\n\n1. **간결하고 명료한 문법**: 파이썬은 코드의 가독성이 뛰어나고 문법이 간결하여 초보자도 쉽게 배우고 사용할 수 있습니다.\n\n2. **다양한 용도**: 웹 개발, 데이터 과학, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다.\n\n3. **광범위한 라이브러리**: 파이썬은 수많은 외부 라이브러리와 프레임워크를 지원하여 개발자들이 특정 작업을 쉽게 수행할 수 있도록 돕습니다. 예를 들어, 데이터 분석을 위한 `pandas`, 머신러닝을 위한 `scikit-learn`, 웹 개발을 위한 `Django` 등이 있습니다.\n\n4. **크로스 플랫폼**: 파이썬은 Windows, macOS, Linux 등 다양한 운영 체제에서 실행될 수 있습니다.\n\n5. **활발한 커뮤니티**: 파이썬은 활발한 개발자 커뮤니티가 있어, 문제 해결을 위한 자료나 도움을 쉽게 찾을 수 있습니다.\n\n이러한 특성 덕분에 파이썬은 초보자부터 전문가까지 널리 사랑받는 언어입니다.', '머신러닝(기계 학습)은 인공지능(AI)의 한 분야로, 컴퓨터가 명시적인 프로그래밍 없이 데이터를 통해 학습하고 경험을 쌓아가는 기술입니다. 머신러닝의 목표는 주어진 데이터를 바탕으로 예측, 분류, 군집화 등의 작업을 수행할 수 있는 모델을 자동으로 성출하는 것입니다.\n\n머신러닝은 대체로 다음과 같은 유형으로 나눌 수 있습니다:\n\n1. **지도학습(Supervised Learning)**: 입력 데이터와 그에 대한 정답(레이블)이 주어진 상태에서 학습합니다. 주어진 데이터를 사용하여 모델을 훈련한 후, 새로운 데이터를 입력했을 때 그에 대한 예측을 할 

/var/folders/k2/q6_f_rp50lqg7l24d_3dk1vh0000gn/T/ipykernel_3701/1733917639.py:11: RuntimeWarning: coroutine 'RunnableSequence.abatch' was never awaited
  res = await chain.abatch(questions)


### 2.3 astream - 비동기 스트리밍

`astream`은 `stream`의 비동기 버전이에요. `async for`로 청크를 하나씩 받아요. 이벤트 루프가 블로킹되지 않아 웹소켓으로 실시간 전송하면서 다른 작업을 동시에 수행할 수 있어요.

In [7]:
# ---------------------------------------------------
# astream(): 비동기 스트리밍 (async for로 청크 수신)
# ---------------------------------------------------
# astream()은 이벤트 루프를 블로킹하지 않아요
# 웹소켓으로 사용자에게 실시간 전송하면서 다른 비동기 작업을 동시에 수행 가능해요

res = chain.astream({"question": "파이썬의 특징 세 가지를 정리하세요"})
async for chunk in res:
    print(chunk, end="", flush=True)

파이썬의 특징 세 가지는 다음과 같습니다:

1. **간결하고 가독성이 높은 코드**: 파이썬은 문법이 간결하고 명확하여 코드의 가독성이 뛰어납니다. 이는 개발자가 코드를 쉽게 이해하고 유지보수할 수 있게 해주며, 학습 곡선이 비교적 완만하여 초보자도 빠르게 익힐 수 있습니다.

2. **광범위한 라이브러리와 프레임워크**: 파이썬은 다양한 분야에서 사용할 수 있는 풍부한 라이브러리와 프레임워크를 제공합니다. 데이터 과학(예: NumPy, Pandas), 웹 개발(예: Django, Flask), 머신러닝(예: TensorFlow, Scikit-learn) 등 다양한 분야에서 필요한 도구를 쉽게 활용할 수 있습니다.

3. **다양한 플랫폼 호환성**: 파이썬은 윈도우, macOS, 리눅스 등 다양한 운영체제에서 실행될 수 있으며, 이식성이 뛰어납니다. 따라서 한 번 코드를 작성하면 여러 플랫폼에서 쉽게 실행할 수 있어 개발 생산성을 높여줍니다.

## 3. 동기 vs 비동기 성능 비교

비동기 메서드를 살펴봤어요. 이제 실제로 시간을 측정해 동기와 비동기 방식의 차이를 직접 확인해볼게요.

> **참고**: `batch()`와 `abatch()`는 모두 내부적으로 최적화가 되어 있어 성능 차이가 크지 않을 수 있어요. 병렬 처리의 이점을 명확히 보려면 개별 `invoke` 순차 호출과 개별 `ainvoke` 병렬 호출을 비교해야 해요.

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| 메서드 | 설명 | 사용 상황 |
|--------|------|-----------|
| `invoke` | 단일 입력, 응답 완료 후 반환 | 단순 단일 호출 |
| `batch` | 여러 입력 일괄 처리 | 다수 입력을 동시에 처리할 때 |
| `stream` | 토큰 단위 실시간 출력 | 챗봇 UI, 긴 응답 실시간 표시 |
| `ainvoke` | 비동기 단일 호출 | 웹 서버, 비동기 환경 |
| `abatch` | 비동기 일괄 처리 | 비동기 환경에서 다수 입력 |
| `astream` | 비동기 스트리밍 | 웹소켓 실시간 전송 |

## 다음 노트북 예고

다음 `04_Runnable_Basic.ipynb`에서는 `RunnablePassthrough`, `RunnableParallel`, `RunnableLambda` 세 가지 핵심 Runnable을 조합해 복잡한 파이프라인을 구성하는 방법을 배울 거예요.

In [ ]:
# ---------------------------------------------------
# 동기 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 코드를 실행하고 순차 invoke vs 병렬 ainvoke 시간 차이를 확인해요
# 힌트: asyncio.gather()로 여러 코루틴을 동시에 실행해요
# 예상 결과: 비동기 병렬 방식이 순차 방식보다 약 2~3배 빠를 거예요
# ============================================================




